# Shor's Algorithm — from period finding to breaking RSA

This notebook is the hands-on companion to the lesson *"RSA and Shor's Algorithm"*.
It walks the full pipeline end to end:

1. **The problem** — order (period) finding of $f(x)=a^x \bmod N$.
2. **The number theory** — why finding the order $r$ lets us factor $N$.
3. **The classical part** — reductions, `gcd`, and (for contrast) brute-force order finding.
4. **The quantum part** — Quantum Phase Estimation (QPE) of the modular-multiplication operator $U$.
5. **Continued fractions** — turning a measured phase $\phi \approx s/r$ into the order $r$.
6. **A complete, robust `shor_factor(N)`** that loops until it succeeds.
7. **Breaking a (toy) RSA key** with the order we recovered — closing the loop with the lesson.
8. **Resource reality check** — why RSA-2048 is still safe *today*.

> Backend: this is the **IQM (Garnet/Emerald on IQM Resonance)** edition.


### Choose where this notebook runs

This is the **IQM edition**: circuits are transpiled to the IQM native gate set **PRX + CZ**
and run on **Garnet** (20 qubits) or **Emerald** (54 qubits) via IQM Resonance. Pick the backend
by editing the single `TARGET` line below (see `README.md` and `iqm_backend.py` in this folder):

| `TARGET` | runs on | token? | QPU credits? |
|---|---|---|---|
| `"fake_apollo"` | local 20-qubit noisy simulator | no | no |
| `"garnet:mock"` / `"emerald:mock"` | IQM Resonance cloud simulator | yes | no |
| `"garnet"` / `"emerald"` | **real QPU** (20q / 54q) | yes | yes |


In [ ]:
from iqm_backend import get_backend, run_counts
TARGET = "fake_apollo"   # change this one line to choose the backend

In [ ]:
# pylint: disable=invalid-name
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import gcd
from fractions import Fraction
from numpy.random import randint

from qiskit import QuantumCircuit
from qiskit.visualization import plot_histogram

print("Imports successful")

## 1. The problem: period (order) finding

Fix two integers $N$ and $a$ with $1 < a < N$ and $\gcd(a,N)=1$. The function

$$ f(x) = a^x \bmod N $$

is **periodic**: there is a smallest positive integer $r$, the **order** of $a$ modulo $N$, with

$$ a^r \equiv 1 \pmod N . $$

Then $f(x+r)=f(x)$ for all $x$. The order $r$ exists because $a$ is invertible modulo $N$
(a consequence of $\gcd(a,N)=1$), so the powers $a^0,a^1,a^2,\dots$ must eventually cycle.

Let's see the period for $a=3,\ N=35$ (the dotted lines only guide the eye).

In [ ]:
N = 35
a = 3

xvals = np.arange(N)
yvals = [pow(a, int(x), N) for x in xvals]

fig, ax = plt.subplots()
ax.plot(xvals, yvals, linewidth=1, linestyle='dotted', marker='x')
ax.set(xlabel='$x$', ylabel=f'${a}^x$ mod ${N}$',
       title="A periodic function in Shor's algorithm")
try:
    r = yvals[1:].index(1) + 1
    plt.annotate('', xy=(0, 1), xytext=(r, 1), arrowprops=dict(arrowstyle='<->'))
    plt.annotate(f'$r={r}$', xy=(r/3, 1.5))
    print(f"Order of {a} mod {N} is r = {r}   (check: {a}^{r} mod {N} = {pow(a, r, N)})")
except ValueError:
    print('Could not find period: check that a < N and gcd(a, N) = 1.')

## 2. Why the order lets us factor $N$

This is the classical reduction that makes Shor's algorithm useful. Suppose we have found the
order $r$ of $a$ modulo $N$, and suppose **$r$ is even**. Write $x = a^{r/2} \bmod N$. Then

$$ x^2 = a^{r} \equiv 1 \pmod N \quad\Longrightarrow\quad x^2 - 1 \equiv 0 \pmod N
   \quad\Longrightarrow\quad (x-1)(x+1) \equiv 0 \pmod N . $$

So $N \mid (x-1)(x+1)$. If in addition **$x \not\equiv -1 \pmod N$** (and clearly $x\not\equiv 1$,
since $r$ is the *smallest* exponent giving $1$), then neither factor is a multiple of $N$ on its own.
Hence $N$ must split its prime factors between $x-1$ and $x+1$, and

$$ \gcd(x-1,\, N) \quad\text{and}\quad \gcd(x+1,\, N) $$

are **non-trivial factors of $N$**. Computing a gcd is cheap (Euclid's algorithm).

**The gamble.** We need $r$ even *and* $a^{r/2}\not\equiv -1$. For a random $a$ coprime to a
(non-prime-power) odd $N$, a classic theorem guarantees this happens with probability
$\ge 1 - 2^{-k}$ where $k$ is the number of distinct odd prime factors of $N$ — so at least
$1/2$, usually $\ge 3/4$. If the gamble fails, we simply pick a new random $a$ and try again.

Everything above is classical. The *only* hard step is computing $r$ — and **that** is what the
quantum computer does exponentially faster than any known classical method.

In [ ]:
def factors_from_order(a, r, N):
    """Given the order r of a mod N, try to extract non-trivial factors of N.
    Returns a sorted list of the non-trivial factors found (possibly empty)."""
    if r % 2 != 0:
        return []                      # r odd: cannot use the (x-1)(x+1) split
    x = pow(a, r // 2, N)
    if x == N - 1:
        return []                      # a^{r/2} == -1 mod N: unlucky, retry with new a
    cands = [gcd(x - 1, N), gcd(x + 1, N)]
    return sorted({c for c in cands if 1 < c < N and N % c == 0})

# Sanity check with the a=3, N=35 example above (r = 12):
print("factors of 35 from a=3, r=12:", factors_from_order(3, 12, 35))

## 3. The classical wrapper (and brute-force order finding for contrast)

A real factoring routine first removes the *easy* cases before ever calling the quantum
subroutine:

* $N$ even $\Rightarrow$ factor $2$.
* $N = c^k$ a **perfect power** $\Rightarrow$ factor $c$ (checkable classically in polynomial time).
* a randomly chosen $a$ that already shares a factor with $N$ ($\gcd(a,N)>1$) is a lucky hit.

Only when none of these apply do we invoke order finding. To *see* what the quantum step
computes, here is a **brute-force** classical order finder — correct but exponentially slow,
exactly the bottleneck Shor removes.

In [ ]:
def classical_order(a, N):
    """Smallest r>0 with a^r = 1 (mod N), found by brute force. O(r) — slow on purpose."""
    assert gcd(a, N) == 1, "a and N must be coprime"
    r, val = 1, a % N
    while val != 1:
        val = (val * a) % N
        r += 1
    return r

def is_perfect_power(N):
    """If N = c**k for integers c>=2, k>=2, return (c, k); else None."""
    for k in range(2, N.bit_length() + 1):
        c = round(N ** (1.0 / k))
        for cc in (c - 1, c, c + 1):
            if cc >= 2 and cc ** k == N:
                return cc, k
    return None

print("classical order of 7 mod 15 :", classical_order(7, 15))   # expect 4
print("is 27 a perfect power?      :", is_perfect_power(27))      # (3, 3)
print("is 15 a perfect power?      :", is_perfect_power(15))      # None

## 4. The quantum part: order finding by Quantum Phase Estimation

Define the unitary modular-multiplication operator

$$ U\,|y\rangle = |a y \bmod N\rangle . $$

Its eigenstates (for $0\le s\le r-1$) are

$$ |u_s\rangle = \frac{1}{\sqrt r}\sum_{k=0}^{r-1} e^{-2\pi i s k / r}\,|a^k \bmod N\rangle,
   \qquad U|u_s\rangle = e^{2\pi i\, s/r}\,|u_s\rangle . $$

The eigenphase is $s/r$ — it **contains the order $r$ we want**. Two facts make this usable:

* The easy-to-prepare state $|1\rangle$ is an equal superposition of all the $|u_s\rangle$:
  $\frac{1}{\sqrt r}\sum_{s=0}^{r-1}|u_s\rangle = |1\rangle$.
* So running **QPE** on $U$ with the register initialised to $|1\rangle$ measures a phase
  $\phi \approx s/r$ for a uniformly random $s\in\{0,\dots,r-1\}$.

We then recover $r$ from $\phi$ with continued fractions (Section 5).

Below, `c_amod15` provides the controlled-$U^{2^q}$ gates for the special case $N=15$ (the standard
small demonstration), and `qft_dagger` is the inverse QFT that reads out the phase.

In [ ]:
def c_amod15(a, power):
    """Controlled multiplication by a mod 15, repeated `power` times (controlled-U^power)."""
    if a not in [2, 4, 7, 8, 11, 13]:
        raise ValueError("'a' must be 2, 4, 7, 8, 11 or 13")
    U = QuantumCircuit(4)
    for _ in range(power):
        if a in [2, 13]:
            U.swap(2, 3); U.swap(1, 2); U.swap(0, 1)
        if a in [7, 8]:
            U.swap(0, 1); U.swap(1, 2); U.swap(2, 3)
        if a in [4, 11]:
            U.swap(1, 3); U.swap(0, 2)
        if a in [7, 11, 13]:
            for q in range(4):
                U.x(q)
    U = U.to_gate()
    U.name = f"{a}^{power} mod 15"
    return U.control()


def qft_dagger(n):
    """Inverse QFT on the first n qubits."""
    qc = QuantumCircuit(n)
    for qubit in range(n // 2):
        qc.swap(qubit, n - qubit - 1)
    for j in range(n):
        for m in range(j):
            qc.cp(-np.pi / float(2 ** (j - m)), m, j)
        qc.h(j)
    qc.name = "QFT†"
    return qc

We use `N_COUNT = 8` counting qubits (more counting qubits $\Rightarrow$ a sharper, more precise
phase estimate). Together with the 4-qubit work register this is a 12-qubit circuit — well within
reach of Garnet (20 qubits) and Emerald (54 qubits).

In [ ]:
def build_order_finding_circuit(a, n_count=8):
    """QPE circuit for the order of a mod 15 with `n_count` counting qubits."""
    qc = QuantumCircuit(n_count + 4, n_count)
    for q in range(n_count):          # counting register -> |+>
        qc.h(q)
    qc.x(n_count)                     # work register -> |1>
    for q in range(n_count):          # controlled-U^{2^q}
        qc.append(c_amod15(a, 2 ** q), [q] + [i + n_count for i in range(4)])
    qc.append(qft_dagger(n_count), range(n_count))   # inverse QFT
    qc.measure(range(n_count), range(n_count))
    return qc

N_COUNT = 8
a = 7
qc = build_order_finding_circuit(a, N_COUNT)
qc.draw('mpl', fold=-1)

Run it and look at the histogram of the counting register.

In [ ]:
counts = run_counts(qc, TARGET, shots=2048)
plot_histogram(counts)

The four tall peaks sit at register values $0, 64, 128, 192$ out of $2^8=256$, i.e. measured
phases $\phi = 0,\ \tfrac14,\ \tfrac12,\ \tfrac34$ — exactly the fractions $s/r$ with $r=4$.

In [ ]:
rows, measured_phases = [], []
for output in counts:
    decimal = int(output, 2)
    phase = decimal / (2 ** N_COUNT)
    measured_phases.append(phase)
    rows.append([f"{output} (bin) = {decimal:>3} (dec)", f"{decimal}/{2**N_COUNT} = {phase:.3f}"])
print(pd.DataFrame(rows, columns=["Register output", "Phase phi = s/r"]).to_string(index=False))

## 5. From phase to order: continued fractions

QPE hands us $\phi \approx s/r$ as a finite-precision binary fraction. We need the *actual*
rational $s/r$ in lowest terms. The **continued fraction expansion** of $\phi$ gives the best
rational approximations with bounded denominator; the theory of QPE guarantees that if the
phase is estimated to enough bits, the convergent with denominator $< N$ is exactly $s/r$.

Python's `fractions.Fraction(...).limit_denominator(N)` does precisely this. Watch why the
`limit_denominator` step is essential — a raw float becomes an ugly exact fraction:

In [ ]:
print("raw            :", Fraction(0.75))                       # ugly / exact binary float
print("limited (<15)  :", Fraction(0.75).limit_denominator(15))  # -> 3/4, denominator r = 4

# For transparency, a from-scratch continued-fraction convergent finder:
def best_rational(phi, max_den):
    """Best rational approx p/q to phi with q <= max_den, via continued fractions."""
    f = Fraction(phi).limit_denominator(max_den)
    return f.numerator, f.denominator

rows = []
for phase in sorted(set(measured_phases)):
    s, r = best_rational(phase, N - 1)      # order must be < N
    rows.append([f"{phase:.3f}", f"{s}/{r}", r])
print(pd.DataFrame(rows, columns=["Phase", "s/r", "Guess for r"]).to_string(index=False))

Phases $\tfrac14$ and $\tfrac34$ both yield $r=4$ (correct). Phase $\tfrac12$ gives the
*factor* $r=2$ of the true order (here $s=1,r=2$ shares no useful split), and phase $0$ (the
$s=0$ outcome) is useless. This is why Shor's algorithm is **probabilistic**: some measurements
fail, so we repeat until the recovered $r$ actually works.

## 6. A complete, robust `shor_factor(N)`

Now we assemble everything: classical reductions $\to$ random $a$ $\to$ quantum order finding
$\to$ continued fractions $\to$ `factors_from_order`, looping until a non-trivial factor appears.
(The quantum step uses the $N=15$ operator, so `shor_factor` is specialised to $N=15$ here — the
same control flow is exactly what a general implementation uses once a general modular-exponentiation
circuit is plugged in; see Section 8.)

In [ ]:
def qpe_order_15(a, n_count=8):
    """One QPE shot -> measured phase for the order of a mod 15 (runs on TARGET)."""
    qc = build_order_finding_circuit(a, n_count)
    counts = run_counts(qc, TARGET, shots=1)      # a single random sample s/r
    reading = next(iter(counts))
    return int(reading, 2) / (2 ** n_count)



def shor_factor(N=15, n_count=8, max_attempts=20, seed=None, verbose=True):
    """Factor N (specialised to N=15 operator) via Shor order finding. Returns (p, q) or None."""
    rng = np.random.default_rng(seed)

    # --- classical shortcuts ---
    if N % 2 == 0:
        return 2, N // 2
    pk = is_perfect_power(N)
    if pk:
        return pk[0], N // pk[0]

    valid_a = [2, 4, 7, 8, 11, 13]            # a coprime to 15 that c_amod15 supports
    for attempt in range(1, max_attempts + 1):
        a = int(rng.choice(valid_a))
        g = gcd(a, N)
        if g > 1:                             # lucky: a already shares a factor
            if verbose: print(f"attempt {attempt}: gcd({a},{N})={g} (lucky hit)")
            return g, N // g
        phase = qpe_order_15(a, n_count)
        s, r = best_rational(phase, N - 1)
        facs = factors_from_order(a, r, N)
        if verbose:
            print(f"attempt {attempt}: a={a}, phi={phase:.3f}, r={r}, factors={facs}")
        if facs:
            p = facs[0]
            return p, N // p
    return None

print("Factoring 15 with Shor's algorithm:")
print("result:", shor_factor(15, seed=2))

## 7. Closing the loop: breaking a (toy) RSA key

In the lesson, RSA security rests on the hardness of factoring the modulus $N=pq$. Here we build a
*tiny* RSA key on $N=15$, then use the factors Shor recovered to reconstruct the **private key** and
decrypt — the exact attack Shor enables (just at a scale small enough to simulate).

In [ ]:
def egcd(a, b):
    if b == 0: return (a, 1, 0)
    g, x, y = egcd(b, a % b)
    return (g, y, x - (a // b) * y)

def modinv(e, m):
    g, x, _ = egcd(e % m, m)
    if g != 1: raise ValueError("no inverse")
    return x % m

# --- toy RSA public key (modulus deliberately factorable) ---
N_rsa = 15
e = 7                       # public exponent, must be coprime to phi(N)
message = 2
cipher = pow(message, e, N_rsa)
print(f"public key (N={N_rsa}, e={e}); ciphertext of m={message} is c={cipher}")

# --- attacker runs Shor to factor N, then rebuilds the private key ---
p, q = shor_factor(N_rsa, seed=2, verbose=False)
phi = (p - 1) * (q - 1)
d = modinv(e, phi)
recovered = pow(cipher, d, N_rsa)
print(f"Shor found p={p}, q={q}  ->  phi={phi}, private key d={d}")
print(f"decrypted message = {recovered}   (attack {'succeeded' if recovered == message else 'failed'})")

## 8. The real bottleneck: modular exponentiation, and a resource reality check

The controlled-$U^{2^q}$ gates are the expensive part. We *cannot* build $U^{2^q}$ by repeating
$U$ a total of $2^q$ times — that is exponential. The trick is that

$$ U^{2^q}|y\rangle = |a^{2^q} y \bmod N\rangle, \qquad a^{2^{q}} \bmod N $$

can be precomputed **classically** by *repeated squaring* in $O(q)$ multiplications, and the modular
multiplier can be realised as a reversible circuit. A clean beginner-friendly construction
(Beauregard) factors an $n$-bit $N$ with only $2n+3$ qubits.

In [ ]:
def a2jmodN(a, j, N):
    """Compute a^{2^j} mod N by repeated squaring (classical, O(j) multiplications)."""
    for _ in range(j):
        a = (a * a) % N
    return a

print("7^(2^2) mod 53 =", a2jmodN(7, 2, 53))      # fast even for huge exponents
print("7^(2^2049) mod 53 =", a2jmodN(7, 2049, 53))

**Why RSA-2048 is still safe today.** Shor's algorithm is *polynomial* — $O((\log N)^3)$ gate
operations with $O(\log N)$ logical qubits — so asymptotically RSA is broken. But the constants and
the **error-correction overhead** are brutal:

* Factoring a 2048-bit modulus needs on the order of a few **thousand logical qubits** and billions
  of fault-tolerant gates (e.g. Gidney–Ekerå estimate: ~20 million *noisy* physical qubits running
  for hours, under plausible assumptions).
* Today's devices — Garnet (20 qubits), Emerald (54 qubits), and the largest superconducting chips
  (hundreds of physical qubits) — are **nowhere near** the fault-tolerant scale required.

So the demonstration here factors $15$ (and with a general modular-exponentiation circuit, numbers
like $21$, $35$), not $2048$-bit keys. The threat is real but *future*, which is exactly why the
world is migrating to **post-quantum cryptography** (lattice-based schemes such as ML-KEM/Kyber).
This mirrors the closing section of the lesson.

## 9. Exercises

1. Re-run the order-finding circuit with $a = 2, 8, 11, 13$. Which orders do you get, and which
   values of $a$ give an *even* order usable for factoring?
2. Change `N_COUNT` to 4 and then to 10. How does the sharpness of the histogram peaks — and the
   reliability of the recovered $r$ — change? Relate this to QPE precision.
3. Run `shor_factor(15)` 50 times with different seeds and estimate the empirical success
   probability per attempt. Compare with the $\ge 1/2$ theoretical bound of Section 2.
4. Using `factors_from_order`, verify by hand (no quantum circuit) that knowing $r=4$ for $a=7$,
   $N=15$ yields the factors $3$ and $5$.

## 10. References

1. P. W. Shor, *Polynomial-Time Algorithms for Prime Factorization and Discrete Logarithms on a
   Quantum Computer*, SIAM J. Comput. **26** (1997) 1484. [arXiv:quant-ph/9508027]
2. S. Beauregard, *Circuit for Shor's algorithm using $2n+3$ qubits*, [arXiv:quant-ph/0205095].
3. M. Nielsen and I. Chuang, *Quantum Computation and Quantum Information*, Cambridge Univ. Press
   (2000), Ch. 5 and p. 633.
4. C. Gidney and M. Ekerå, *How to factor 2048 bit RSA integers in 8 hours using 20 million noisy
   qubits*, Quantum **5** (2021) 433. [arXiv:1905.09749]
